# Cell 1: Import Dependencies

In [1]:
import re
import os
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import pyarrow.dataset as ds

# Cell 2: Configuration

In [2]:
# ===== Input and output paths =====
INPUT_FILE = "/home/jovyan/data/fineweb/004_00018.parquet"

OUTPUT_DIR = "cleaning_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLEAN_FILE = os.path.join(OUTPUT_DIR, "004_00049.clean.parquet")
REMOVED_FILE = os.path.join(OUTPUT_DIR, "004_00049.removed.parquet")
METRICS_FILE = os.path.join(OUTPUT_DIR, "004_00049.metrics.csv")
RULE_COUNTS_FILE = os.path.join(OUTPUT_DIR, "004_00049.rule_counts.csv")
RAW_SAMPLE_FILE = os.path.join(OUTPUT_DIR, "004_00049.raw_sample50.csv")
CLEAN_SAMPLE_FILE = os.path.join(OUTPUT_DIR, "004_00049.clean_sample50.csv")

# ===== Recommended thresholds (adjust as needed) =====
MIN_TOKEN_COUNT = 80
MIN_LANGUAGE_SCORE = 0.90
TARGET_LANGUAGE = "en"

# Suspicious URL patterns
SUSPICIOUS_URL_PATTERNS = [
    r"/tag/",
    r"/tags/",
    r"/category/",
    r"/author/",
    r"/page/\d+",
    r"\?replytocom=",
    r"\?amp",
    r"/wp-content/",
]

# Low-information / template-like keywords
LOW_INFO_PATTERNS = [
    r"\bprivacy policy\b",
    r"\bterms of service\b",
    r"\bcookie policy\b",
    r"\ball rights reserved\b",
    r"\bskip to content\b",
    r"\bclick here\b",
    r"\bsubscribe\b",
    r"\bsign up\b",
    r"\blog in\b",
    r"\bmenu\b",
    r"\bnavigation\b",
    r"\bhome\b",
    r"\bcontact us\b",
]

# Suspicious domain blacklist (leave empty for now; add domains later if needed)
BAD_DOMAINS = set()

# Cell 3: Load Data

In [3]:
dataset = ds.dataset(INPUT_FILE, format="parquet")
df = dataset.to_table().to_pandas()

print("rows:", len(df))
print("columns:", list(df.columns))
df.head(3)

rows: 172447
columns: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count']


,text,id,dump,url,date,file_path,language,language_score,token_count
0,Welcome to Harrisonburg TownSquare.\nTownSquar...,<urn:uuid:189514cf-784a-4e9f-9e8e-f350c5a41adf>,CC-MAIN-2025-26,https://www.townsquareapp.co/,2025-06-16T16:22:40Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.899284,203
1,"The Art of Effective Board Reporting\nSep 5, 2...",<urn:uuid:9d2f701d-48f5-418f-a8e2-d5b2d51aad36>,CC-MAIN-2025-26,https://www.traact.com/blog/the-art-of-effecti...,2025-06-16T16:45:14Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.942698,2717
2,Belinda's journey and transformation is a real...,<urn:uuid:ccb16b4e-f301-480c-9231-33f6e823eabe>,CC-MAIN-2025-26,https://www.tracydixonmindandbodyfit.com/post/...,2025-06-16T15:44:36Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.987144,555


# Cell 4: Helper Functions

In [4]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\x00", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def get_domain(url):
    try:
        return urlparse(str(url)).netloc.lower()
    except:
        return ""

def is_empty_or_almost_empty(text):
    if not text:
        return True
    # If little content remains after removing non-alphanumeric characters, treat it as empty / near-empty
    core = re.sub(r"[^A-Za-z0-9]+", "", text)
    return len(core) < 20

def has_suspicious_url(url):
    url = str(url).lower()
    for pat in SUSPICIOUS_URL_PATTERNS:
        if re.search(pat, url):
            return True
    return False

def repeated_char_ratio(text):
    # Maximum character frequency divided by text length; a rough signal for abnormal character repetition
    if not text:
        return 1.0
    chars = [c for c in text if not c.isspace()]
    if not chars:
        return 1.0
    from collections import Counter
    cnt = Counter(chars)
    return max(cnt.values()) / len(chars)

def repeated_ngram_ratio(text, n=3):
    # Rough detection of repeated n-grams
    words = text.lower().split()
    if len(words) < n * 2:
        return 0.0
    ngrams = [" ".join(words[i:i+n]) for i in range(len(words) - n + 1)]
    if not ngrams:
        return 0.0
    from collections import Counter
    cnt = Counter(ngrams)
    return max(cnt.values()) / len(ngrams)

def is_low_information(text):
    text_l = text.lower()
    # Keyword-based trigger
    keyword_hits = sum(bool(re.search(pat, text_l)) for pat in LOW_INFO_PATTERNS)
    # Repeated-character / repeated-ngram trigger
    char_rep = repeated_char_ratio(text)
    ngram_rep = repeated_ngram_ratio(text, n=3)
    # A rough proxy for information density: very low unique-word ratio
    words = re.findall(r"[A-Za-z]+", text_l)
    uniq_ratio = len(set(words)) / len(words) if words else 0.0

    return (
        keyword_hits >= 3
        or char_rep > 0.35
        or ngram_rep > 0.08
        or (len(words) >= 30 and uniq_ratio < 0.22)
    )

def exact_text_key(text):
    # Use normalized text as an exact deduplication key
    return normalize_text(text).lower()

# Cell 5: Feature Preparation

In [5]:
df = df.copy()

df["text_norm"] = df["text"].apply(normalize_text)
df["domain"] = df["url"].apply(get_domain)

# Standardize column types to avoid issues in null checks
for col in ["id", "dump", "url", "date", "file_path", "language"]:
    df[col] = df[col].astype("string")

df["language_score"] = pd.to_numeric(df["language_score"], errors="coerce")
df["token_count"] = pd.to_numeric(df["token_count"], errors="coerce")

df[["text_norm", "domain", "language", "language_score", "token_count"]].head(3)

,text_norm,domain,language,language_score,token_count
0,Welcome to Harrisonburg TownSquare. TownSquare...,www.townsquareapp.co,en,0.899284,203
1,"The Art of Effective Board Reporting Sep 5, 20...",www.traact.com,en,0.942698,2717
2,Belinda's journey and transformation is a real...,www.tracydixonmindandbodyfit.com,en,0.987144,555


# Cell 6: Mark Removal Reasons

In [6]:
remove_reasons = []

# Define boolean filtering rules
rule_empty_text = df["text_norm"].apply(is_empty_or_almost_empty)
rule_short_text = df["token_count"].fillna(0) < MIN_TOKEN_COUNT
rule_non_english = df["language"].fillna("").str.lower() != TARGET_LANGUAGE
rule_low_lang_score = df["language_score"].fillna(0) < MIN_LANGUAGE_SCORE
rule_bad_domain = df["domain"].isin(BAD_DOMAINS)
rule_suspicious_url = df["url"].fillna("").astype(str).apply(has_suspicious_url)
rule_low_info = df["text_norm"].apply(is_low_information)

# Deduplication rules
rule_dup_id = df["id"].duplicated(keep="first")
rule_dup_url = df["url"].duplicated(keep="first")
text_keys = df["text_norm"].apply(exact_text_key)
rule_dup_text = text_keys.duplicated(keep="first")

rule_map = {
    "empty_or_almost_empty_text": rule_empty_text,
    "token_count_too_low": rule_short_text,
    "non_english_language": rule_non_english,
    "language_score_too_low": rule_low_lang_score,
    "bad_domain": rule_bad_domain,
    "suspicious_url_pattern": rule_suspicious_url,
    "low_information_or_template_page": rule_low_info,
    "duplicate_id": rule_dup_id,
    "duplicate_url": rule_dup_url,
    "duplicate_text_exact": rule_dup_text,
}

# Combine all matched rules for each row into a single string
reason_cols = []
for rule_name, mask in rule_map.items():
    col_name = f"rm_{rule_name}"
    df[col_name] = mask
    reason_cols.append(col_name)

def collect_reasons(row):
    reasons = []
    for rule_name in rule_map.keys():
        if row[f"rm_{rule_name}"]:
            reasons.append(rule_name)
    return "|".join(reasons)

df["remove_reasons"] = df.apply(collect_reasons, axis=1)
df["is_removed"] = df["remove_reasons"] != ""

print(df["is_removed"].value_counts())

is_removed
False    142316
True      30131
Name: count, dtype: int64


# Cell 7: Build Cleaned and Removed Datasets

In [7]:
clean_df = df.loc[~df["is_removed"]].copy()
removed_df = df.loc[df["is_removed"]].copy()

print("raw rows   :", len(df))
print("clean rows :", len(clean_df))
print("removed    :", len(removed_df))
print("remove rate:", len(removed_df) / len(df))

raw rows   : 172447
clean rows : 142316
removed    : 30131
remove rate: 0.17472614774394452


# Cell 8: Evaluation Function

In [8]:
def dataset_metrics(x: pd.DataFrame, name: str):
    out = {}
    out["dataset"] = name
    out["doc_count"] = len(x)

    out["avg_token_count"] = x["token_count"].mean()
    out["median_token_count"] = x["token_count"].median()
    out["p25_token_count"] = x["token_count"].quantile(0.25)
    out["p75_token_count"] = x["token_count"].quantile(0.75)
    out["p90_token_count"] = x["token_count"].quantile(0.90)

    out["short_lt_50_ratio"] = (x["token_count"] < 50).mean()
    out["short_lt_100_ratio"] = (x["token_count"] < 100).mean()

    out["english_ratio"] = (x["language"].fillna("").str.lower() == "en").mean()
    out["avg_language_score"] = x["language_score"].mean()
    out["median_language_score"] = x["language_score"].median()
    out["lang_score_lt_08_ratio"] = (x["language_score"] < 0.8).mean()
    out["lang_score_lt_09_ratio"] = (x["language_score"] < 0.9).mean()

    out["duplicate_id_ratio"] = x["id"].duplicated().mean()
    out["duplicate_url_ratio"] = x["url"].duplicated().mean()
    out["duplicate_text_ratio"] = x["text_norm"].apply(exact_text_key).duplicated().mean()

    out["unique_domain_count"] = x["domain"].nunique()
    return out

# Cell 9: Compute Before/After Quality Metrics

In [9]:
raw_metrics = dataset_metrics(df, "raw")
clean_metrics = dataset_metrics(clean_df, "clean")

metrics_df = pd.DataFrame([raw_metrics, clean_metrics])

# Add retention and deletion ratios
raw_count = len(df)
clean_count = len(clean_df)

metrics_df["retention_ratio"] = metrics_df["doc_count"] / raw_count
metrics_df["deletion_ratio"] = 1 - metrics_df["retention_ratio"]

metrics_df

,dataset,doc_count,avg_token_count,median_token_count,p25_token_count,p75_token_count,p90_token_count,short_lt_50_ratio,short_lt_100_ratio,english_ratio,avg_language_score,median_language_score,lang_score_lt_08_ratio,lang_score_lt_09_ratio,duplicate_id_ratio,duplicate_url_ratio,duplicate_text_ratio,unique_domain_count,retention_ratio,deletion_ratio
0,raw,172447,779.813833,487.0,241.0,916.0,1611.0,0.000429,0.039392,1.0,0.935297,0.943012,0.012601,0.139452,0.0,0.000087,0.0,139653,1.000000,0.000000
1,clean,142316,774.271923,530.0,276.0,957.0,1620.0,0.000000,0.018951,1.0,0.947389,0.948491,0.000000,0.000000,0.0,0.000000,0.0,115465,0.825274,0.174726


# Cell 10: Count Removals by Rule

In [10]:
rule_counts = []
for rule_name in rule_map.keys():
    cnt = int(rule_map[rule_name].sum())
    rule_counts.append({
        "rule": rule_name,
        "removed_count": cnt,
        "removed_ratio_over_raw": cnt / len(df)
    })

rule_counts_df = pd.DataFrame(rule_counts).sort_values("removed_count", ascending=False)
rule_counts_df

,rule,removed_count,removed_ratio_over_raw
3,language_score_too_low,24048,0.139452
5,suspicious_url_pattern,3676,0.021317
1,token_count_too_low,2516,0.014590
6,low_information_or_template_page,1809,0.010490
8,duplicate_url,15,0.000087
0,empty_or_almost_empty_text,0,0.000000
4,bad_domain,0,0.000000
2,non_english_language,0,0.000000
7,duplicate_id,0,0.000000
9,duplicate_text_exact,0,0.000000


# Cell 11: Source Quality Comparison (Top Domains)

In [11]:
raw_domain_top = df["domain"].value_counts().head(20).rename("raw_top_domains")
clean_domain_top = clean_df["domain"].value_counts().head(20).rename("clean_top_domains")

print("=== Raw top domains ===")
display(raw_domain_top)

print("\n=== Clean top domains ===")
display(clean_domain_top)

=== Raw top domains ===


domain
pubmed.ncbi.nlm.nih.gov        59
plandisney.disney.go.com       53
www.telegraph.co.uk            46
www.ibtimes.co.uk              43
www.cbsnews.com                42
www.latimes.com                38
www.sciencedaily.com           38
www.prweb.com                  38
timesofindia.indiatimes.com    37
publications.parliament.uk     33
www.techradar.com              32
www.foxnews.com                32
www.gofundme.com               32
community.oracle.com           29
www.csmonitor.com              29
www.aljazeera.com              29
dev.to                         28
www.jalopnik.com               27
www.digitaltrends.com          27
forum.trains.com               26
Name: raw_top_domains, dtype: int64


=== Clean top domains ===


domain
plandisney.disney.go.com       50
www.telegraph.co.uk            46
pubmed.ncbi.nlm.nih.gov        43
www.ibtimes.co.uk              43
www.cbsnews.com                41
www.latimes.com                38
www.sciencedaily.com           38
timesofindia.indiatimes.com    37
www.prweb.com                  36
www.foxnews.com                32
www.gofundme.com               32
publications.parliament.uk     31
www.csmonitor.com              29
www.aljazeera.com              28
www.techradar.com              27
www.jalopnik.com               27
www.digitaltrends.com          27
forum.trains.com               26
www.themoscowtimes.com         25
www.business-standard.com      25
Name: clean_top_domains, dtype: int64

# Cell 12: Export Manual Review Samples (50 Each)

In [12]:
raw_sample = df.sample(min(50, len(df)), random_state=42)[
    ["text", "url", "domain", "language", "language_score", "token_count"]
].copy()

clean_sample = clean_df.sample(min(50, len(clean_df)), random_state=42)[
    ["text", "url", "domain", "language", "language_score", "token_count"]
].copy()

# Columns for later manual annotation
for sample_df in [raw_sample, clean_sample]:
    sample_df["manual_label"] = ""   # normal article / low-information page / ad page / navigation-template page / junk text / suspicious content
    sample_df["manual_notes"] = ""

raw_sample.to_csv(RAW_SAMPLE_FILE, index=False)
clean_sample.to_csv(CLEAN_SAMPLE_FILE, index=False)

display(raw_sample.head(5))
display(clean_sample.head(5))

,text,url,domain,language,language_score,token_count,manual_label,manual_notes
88554,Michigan and Detroit agree on deal to help cit...,https://www.marketplace.org/story/2012/04/05/m...,www.marketplace.org,en,0.959616,371,,
93333,Hello Kitty Dressing Set For Girls\nLength 27 ...,https://kiddyland.pk/products/barbie-mirror-fo...,kiddyland.pk,en,0.743098,76,,
154498,Navigating the Journey from Caregiver to Survi...,https://www.va.gov/outreach-and-events/events/...,www.va.gov,en,0.944833,237,,
60880,Graduation of the First Cohort of the Master o...,https://www.alfozan.com/press-releases/tkhryj-...,www.alfozan.com,en,0.933934,279,,
35722,Looking for top attractions in Riyadh around t...,https://www.hyattbuyutat-riyadh.website/top-at...,www.hyattbuyutat-riyadh.website,en,0.967340,1083,,


,text,url,domain,language,language_score,token_count,manual_label,manual_notes
146414,Welcome to Storgage\nYour one stop solution fo...,https://storgage.com/,storgage.com,en,0.939487,793,,
665,The 12th of February marks three years since t...,https://icds.ee/en/minsk-deadlock-three-years-...,icds.ee,en,0.958927,1650,,
172084,The marketing landscape is evolving continuous...,https://www.launchnotes.com/blog/top-10-produc...,www.launchnotes.com,en,0.932553,2541,,
20360,How to Build the Perfect Timber Deck in Austra...,https://hardwoodmills.com.au/how-to-build-the-...,hardwoodmills.com.au,en,0.923002,670,,
16036,14 December 2020\nHistory has beautifully repe...,https://www.shcj.org/twin-sisters-profess-perp...,www.shcj.org,en,0.978312,852,,


# Cell 13: Save Outputs

In [13]:
# Drop intermediate helper columns before saving
helper_cols = ["text_norm", "domain", "remove_reasons", "is_removed"] + reason_cols

save_clean_df = clean_df.drop(columns=[c for c in helper_cols if c in clean_df.columns], errors="ignore")
save_removed_df = removed_df.copy()

save_clean_df.to_parquet(CLEAN_FILE, index=False)
save_removed_df.to_parquet(REMOVED_FILE, index=False)
metrics_df.to_csv(METRICS_FILE, index=False)
rule_counts_df.to_csv(RULE_COUNTS_FILE, index=False)

print("Saved:")
print(" clean data   ->", CLEAN_FILE)
print(" removed data ->", REMOVED_FILE)
print(" metrics      ->", METRICS_FILE)
print(" rule counts  ->", RULE_COUNTS_FILE)
print(" raw sample   ->", RAW_SAMPLE_FILE)
print(" clean sample ->", CLEAN_SAMPLE_FILE)

Saved:
 clean data   -> cleaning_outputs/004_00049.clean.parquet
 removed data -> cleaning_outputs/004_00049.removed.parquet
 metrics      -> cleaning_outputs/004_00049.metrics.csv
 rule counts  -> cleaning_outputs/004_00049.rule_counts.csv
 raw sample   -> cleaning_outputs/004_00049.raw_sample50.csv
 clean sample -> cleaning_outputs/004_00049.clean_sample50.csv


# Cell 14: Print Summary

In [14]:
print("=== Summary ===")
print(f"Raw docs:   {len(df):,}")
print(f"Clean docs: {len(clean_df):,}")
print(f"Removed:    {len(removed_df):,}")
print(f"Retention:  {len(clean_df)/len(df):.2%}")
print(f"Deletion:   {len(removed_df)/len(df):.2%}")

print("\n=== Main metrics comparison ===")
display(metrics_df)

print("\n=== Rule impact ===")
display(rule_counts_df)

=== Summary ===
Raw docs:   172,447
Clean docs: 142,316
Removed:    30,131
Retention:  82.53%
Deletion:   17.47%

=== Main metrics comparison ===


,dataset,doc_count,avg_token_count,median_token_count,p25_token_count,p75_token_count,p90_token_count,short_lt_50_ratio,short_lt_100_ratio,english_ratio,avg_language_score,median_language_score,lang_score_lt_08_ratio,lang_score_lt_09_ratio,duplicate_id_ratio,duplicate_url_ratio,duplicate_text_ratio,unique_domain_count,retention_ratio,deletion_ratio
0,raw,172447,779.813833,487.0,241.0,916.0,1611.0,0.000429,0.039392,1.0,0.935297,0.943012,0.012601,0.139452,0.0,0.000087,0.0,139653,1.000000,0.000000
1,clean,142316,774.271923,530.0,276.0,957.0,1620.0,0.000000,0.018951,1.0,0.947389,0.948491,0.000000,0.000000,0.0,0.000000,0.0,115465,0.825274,0.174726



=== Rule impact ===


,rule,removed_count,removed_ratio_over_raw
3,language_score_too_low,24048,0.139452
5,suspicious_url_pattern,3676,0.021317
1,token_count_too_low,2516,0.014590
6,low_information_or_template_page,1809,0.010490
8,duplicate_url,15,0.000087
0,empty_or_almost_empty_text,0,0.000000
4,bad_domain,0,0.000000
2,non_english_language,0,0.000000
7,duplicate_id,0,0.000000
9,duplicate_text_exact,0,0.000000
